# ДЗ 2. Функции и словари на настоящих данных
Срок: **до понедельника 19:00**.

**Сдача через GitHub** (инструкция «Git для домашних заданий»): сохраните ноутбук как `hw02.ipynb` в папку `hw02` репозитория `dsa-homework`, затем `git add hw02` → `git commit -m "hw02: ..."` → `git push`. Из Colab: Файл → Скачать → Скачать .ipynb. Файл `orders.csv` в репозиторий класть не нужно.

Файл `orders.csv` — заказы «Дала Маркет» за сентябрь и октябрь 2026. Работаем **без Pandas и NumPy**: только `csv`, функции, словари, множества.

- **[База]** обязательно для всех.
- **[Стандарт]** решайте, сколько получится.
- **[Челлендж]** для тех, кому легко.

**Правило AI:** можно спросить, *почему* не работает код, но нельзя просить решение.

## [Для всех] Чтение кода — без запуска
Для каждого фрагмента: **сначала** напишите в Markdown-ячейке ответ и одно предложение «почему», **потом** запустите и сравните. Ошибка в предсказании не снижает оценку — снижает отсутствие объяснения.
```python
# 1
r = {}
for c in ["A", "B", "A"]:
    r[c] = r.get(c, 0) + 1
print(r)

# 2
def f(x):
    x.append(1)
a = [0]; f(a); f(a); print(a)

# 3
s = {c.lower() for c in ["Almaty", "ALMATY", "Astana"]}; print(len(s))

# 4
d = {k: len(k) for k in ["ab", "abc", "ab"]}; print(d)

# 5
def g(n):
    if n > 0:
        print(n)
x = g(5); print(x)
```
На следующем занятии несколько человек объяснят вслух одну строку своего ДЗ.

Мои предсказания и объяснения:

1. Предсказание: {'A': 2, 'B': 1}
- Почему: r.get(c, 0) возвращает текущий счётчик или 0, если ключа ещё нет, поэтому "A" встречается дважды и получает 2
2. Предсказание: [0, 1, 1]
- Почему: список изменяемый и в функцию передаётся ссылка на тот же объект, поэтому два вызова f(a) добавляют по единице в исходный a
3. Предсказание: 2
- Почему: lower() превращает "Almaty" и "ALMATY" в одну строку "almaty", а множество хранит только уникальные элементы, поэтому остаётся {"almaty", "astana"}
4. Предсказание: {'ab': 2, 'abc': 3}
- Почему: ключи словаря уникальны, поэтому второе "ab" перезаписывает первое тем же значением 2 и в словаре остаётся две записи
5. Предсказание: сначала 5, потом None
- Почему: g(5) печатает 5 внутри функции, а return в ней нет, поэтому функция возвращает None, и именно оно попадает в x

In [ ]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

In [1]:
import csv
with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
print(len(rows), "строк")

5463 строк


## [База]
1. Напишите функции `to_num(s)` и `clean_city(s)` (6 городов, латиница и разный регистр → кириллица с заглавной).
2. Сколько заказов в каждой категории (`category`)? Отсортируйте по убыванию.
3. Для каждой категории посчитайте выручку оплаченных заказов и средний чек. Выведите таблицу f-строкой.
4. Сколько заказов в каждом месяце? Месяц — это `order_date[:7]`.

In [10]:
import csv

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
print(len(rows), "строк")

# 1
def to_num(s):
    s = "".join(s.split())
    try:
        return float(s)
    except ValueError:
        return None

CITIES = {"алматы": "Алматы", "almaty": "Алматы",
          "астана": "Астана", "astana": "Астана",
          "шымкент": "Шымкент", "shymkent": "Шымкент",
          "караганда": "Караганда", "актобе": "Актобе", "атырау": "Атырау"}

def clean_city(s):
    key = s.strip().lower()
    return CITIES.get(key, s.strip().capitalize())

for r in rows:
    r["city"] = clean_city(r["city"])
    r["amount"] = to_num(r["amount"])

# 2
counts = {}
for r in rows:
    counts[r["category"]] = counts.get(r["category"], 0) + 1

print("\nЗаказов по категориям:")
for cat, n in sorted(counts.items(), key=lambda kv: -kv[1]):
    print(f"  {cat:<12} {n:>5}")

# 3
revenue = {}   
count = {}    

for r in rows:
    if r["status"] != "оплачен":      
        continue
    if r["amount"] is None:          
        continue
    cat = r["category"]
    if cat not in revenue:       
        revenue[cat] = 0
        count[cat] = 0
    revenue[cat] = revenue[cat] + r["amount"]
    count[cat] = count[cat] + 1

print("\nОплаченные заказы:")
print(f"  {'категория':<12} {'заказов':>8} {'выручка':>14} {'ср. чек':>10}")
categories = sorted(revenue, key=revenue.get, reverse=True)
for cat in categories:
    avg = revenue[cat] / count[cat]   
    print(f"  {cat:<12} {count[cat]:>8} {revenue[cat]:>14,.0f} {avg:>10,.0f}")

# 4
months = {}
for r in rows:
    m = r["order_date"][:7]
    months[m] = months.get(m, 0) + 1

print("\nЗаказов по месяцам:")
for m in sorted(months):
    print(f"  {m}  {months[m]:>5}")

5463 строк

Заказов по категориям:
  Одежда        1529
  Электроника   1185
  Дом           1105
  Красота        991
  Книги          653

Оплаченные заказы:
  категория     заказов        выручка    ср. чек
  Электроника      1065     65,270,240     61,287
  Одежда           1379     18,321,080     13,286
  Дом               987     10,516,320     10,655
  Красота           890      5,507,620      6,188
  Книги             583      2,294,100      3,935

Заказов по месяцам:
  2026-09   2782
  2026-10   2681


## [Стандарт]
5. Выручка оплаченных по месяцам и изменение октября к сентябрю в процентах.
6. Доля отменённых заказов по каждому (чистому) городу, в процентах с одним знаком.
7. Топ-5 клиентов (`customer_id`) по выручке оплаченных заказов. Кто на первом месте и что с ним не так? Ответьте в Markdown-ячейке.
8. Сколько **уникальных** клиентов в каждом городе? Подсказка: словарь множеств.

In [ ]:
# Стандарт
# 5


# 6


# 7


# 8



## [Челлендж]
9. Найдите полные дубли строк: превратите каждую строку в `tuple(r.values())` и проверяйте через множество. Сколько их и в каком месяце?
10. Пересчитайте выручку по месяцам без дублей и без клиента из задачи 7. Как изменился процент из задачи 5?
11. У какого клиента больше всего заказов (после удаления дублей)? Сколько?
12. Напишите менеджеру 3–4 предложения: насколько на самом деле упала выручка и почему первый расчёт был неверным.

In [ ]:
# Челлендж